In [ ]:
import pandas as pd
from pathlib import Path

DATA = Path("..") / "data"
RAW = DATA / "raw"
PROCESSED = DATA / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)   # raw is never created by code — it's where downloads live

df = pd.read_csv(
    RAW / "mrds-GA.csv",
    low_memory=False,
    encoding="latin-1",
)

print(df.shape)
print(df.columns.tolist())

keep = ["dep_id", "site_name", "latitude", "longitude", "commod1"]
df = df[keep].dropna(subset=["latitude", "longitude"])

print(df["commod1"].value_counts().head(20))

df.head(10)


FileNotFoundError: [Errno 2] No such file or directory: '..\\data\\mrds-GA.csv'

In [ ]:
import requests
import time
import pandas as pd

MACROSTRAT_URL = "https://macrostrat.org/api/geologic_units/map"

def get_geology(lat: float, lng: float) -> dict:
    """Query Macrostrat for the geologic unit at a point.
    Prefers the most detailed (largest-scale) map available."""
    try:
        r = requests.get(
            MACROSTRAT_URL,
            params={"lat": lat, "lng": lng, "scale": "large"},
            timeout=10,
        )
        r.raise_for_status()
        data = r.json()["success"]["data"]

        # fallback
        if not data:
            r = requests.get(
                MACROSTRAT_URL,
                params={"lat": lat, "lng": lng},
                timeout=10,
            )
            r.raise_for_status()
            data = r.json()["success"]["data"]

        if not data:
            return {}  # no map coverage

        # want smallest span, Decision #2
        best = min(data, key=lambda u: (u["b_age"] or 4600) - (u["t_age"] or 0))

        return {
            "lith": best.get("lith"),
            "b_age": best.get("b_age"),
            "t_age": best.get("t_age"),
            "unit_name": best.get("name"),
            "map_source": best.get("source_id"),
        }
    except Exception as e:
        print(f"  miss at ({lat:.3f}, {lng:.3f}): {e}")
        return {}

sample = df.sample(25, random_state=42).reset_index(drop=True)

rows = []
for i, rec in sample.iterrows():
    geo = get_geology(rec["latitude"], rec["longitude"])
    rows.append({
        "dep_id": rec["dep_id"],
        "site_name": rec["site_name"],
        "commodity": rec["commod1"],
        "lat": rec["latitude"],
        "lng": rec["longitude"],
        **geo,
    })
    print(f"{i+1:>2}/25  {str(rec['site_name'])[:35]:<35} -> {geo.get('unit_name', 'NO DATA')}")
    time.sleep(0.5)

day_one = pd.DataFrame(rows)
day_one.to_csv(DATA / "day_one_table.csv", index=False)
day_one